# SeismicData Assessment

**Run this notebook in Google Colab.**

The application code is intentionally implemented in `src/seismicdata/`
so the evaluator can inspect normal Python modules. This notebook is the
reproducible entry point.

## 1. Install dependencies

If this notebook was opened from the GitHub repository, the repository
should already be the current directory.

In [ ]:
from pathlib import Path
import os

REPO = Path("/content/seismicdata_assessment")
if REPO.exists():
    os.chdir(REPO)

print("Working directory:", Path.cwd())

In [ ]:
!pip install -q -r requirements.txt

## 2. Verify evaluator MiniSEED files

Expected:

- `SEP01.mseed`
- `SEP02.mseed`
- `SEP03.mseed`

in `data/SEP/`.

In [ ]:
from pathlib import Path

data_dir = Path("data/SEP")

files = sorted(data_dir.glob("*.mseed"))

if not files:
    print("No evaluator MiniSEED files found.")
    print("Place SEP01.mseed, SEP02.mseed and SEP03.mseed in data/SEP/.")
else:
    for file in files:
        print(f"{file.name}: {file.stat().st_size:,} bytes")

## 3. Run the complete pipeline

In [ ]:
!python run_pipeline.py --data-dir data/SEP --output-dir output

## 4. Display generated outputs

In [ ]:
from pathlib import Path
from IPython.display import display, Image, HTML

output = Path("output")

print("Generated files:")
for p in sorted(output.iterdir()):
    print(" -", p.name)

for image in sorted(output.glob("helicorder_*.png")):
    display(Image(filename=str(image)))

if (output / "seismic_dashboard.html").exists():
    display(HTML(
        "<p><b>Dashboard:</b> "
        "<a href='output/seismic_dashboard.html' target='_blank'>"
        "Open dashboard</a></p>"
    ))

## 5. Inspect database

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("output/seismicdata.sqlite")

display(pd.read_sql_query(
    "SELECT * FROM network ORDER BY code", conn
))

display(pd.read_sql_query(
    "SELECT * FROM station ORDER BY network_id, code", conn
))

display(pd.read_sql_query(
    "SELECT * FROM waveform_segment LIMIT 20", conn
))

display(pd.read_sql_query(
    "SELECT * FROM sample LIMIT 20", conn
))

## 6. Inspect required HOA and SUG stations

In [ ]:
display(pd.read_sql_query(
    """
    SELECT n.code AS network,
           s.code AS station,
           s.latitude,
           s.longitude,
           s.elevation_m,
           s.site_name
    FROM station s
    JOIN network n ON n.network_id = s.network_id
    WHERE s.code IN ('HOA', 'SUG')
    ORDER BY s.code
    """,
    conn
))